# 12.3 短聲音如何分成重疊的時間框？


一段 1600 點、0.1 秒的單音，想分段看局部變化。取前 400 點作第一框，往右移 160 點再取 400 點作第二框；相鄰兩框重疊 240 點，同一段聲音會在兩框中出現。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-12-frames.svg")))

In [ ]:
import torch
from tiny_perceptron.multimodal import tone

wave = tone()
frames = wave.unfold(0, 400, 160)
print("框形狀", tuple(frames.shape))
print("框秒數", 400 / 16000, "移動秒數", 160 / 16000)
print("重疊相同", torch.equal(frames[0, 160:], frames[1, :240]))

`unfold(0,400,160)` 沿唯一的時間軸取窗，不改振幅。輸出 `(8,400)`，每框 400／16000＝0.025 秒，框起點每次前進 0.01 秒。比較第一框後 240 點與第二框前 240 點，印 True，直接核對它們來自同一段原資料。

八框起點為 0、160、…、1120；最後一框到 1519。末尾 80 點不足再放一個完整框，這個 `unfold` 例子便留在框外。若補零或先在兩端加資料，框數會不同；後面的頻譜函式有自己的邊界約定，不能只看「400、160」就以為所有框數必相同。

短框方便表示局部頻率，但框太短時很難分清接近的頻率；框太長又把較長時間混在一起。重疊能讓起點落在框邊的聲音仍出現在鄰框中，代價是更多計算，不是新增一份獨立錄音。

資料拆分也要守住這點：同一錄音的所有時間框應跟原錄音在同一分組，不能把第一框訓練、第二框測試，便稱測到新聲音。練習把步長改為 400，完整框數應為 4 且沒有重疊。

<details>
<summary>回顧與查證</summary>

可回顧：[12.2樣本到時間的換算](https://birdhackor.github.io/tiny-perceptron-vlm/12.2.html)。

</details>
